# Backtest auditável — Robo Abertura WDO v1.35

Notebook de **pesquisa, validação e relatório**. Toda a lógica (dados, indicadores, motor, métricas) vive no pacote `wdo` (`src/wdo/`); aqui apenas se configura o cenário, executa e se visualizam os resultados. Não é equivalente tick-a-tick ao Strategy Tester: usa candles M5 e a política OHLC conservadora configurável.


## 1. Overview e auditoria da estratégia

**Inputs do EA preservados:** símbolo, lote, gain/loss, offset e modo de canal, rompimento, magic, desvio, tolerância EMA, RSI(7), faixas RSI e janela 09:00–10:30.

**Indicadores:** EMA 13/17/21 em H1 e D1; RSI(7) M5; PDH/PDL do dia anterior. O EA consulta EMAs H1/D1 na barra ainda aberta. Este backtest usa o último valor disponível antes da barra M5, sem look-ahead.

| Regra MT5 | Implementação Python |
|---|---|
| P1: toque da primeira EMA | extrema OHLC toca a zona EMA; entrada no open da próxima oportunidade |
| P2: mínima/máxima da vela anterior | referência móvel, processada barra a barra |
| P3: stop/fade no canal | ordens pendentes stop/limit; primeira execução vence |
| P4: RSI fora de PDH/PDL | faixas e espera traduzidas para RSI M5 conhecido |
| TP/SL 6/10 pontos | níveis por trade, com política intrabar conservadora |
| uma operação/dia e fim de janela | estado diário e cancelamento de pendências |

### Limitações deliberadas
- Não há ticks, bid/ask ou book: spread deve ser incorporado em `slippage_points`.
- Se stop e alvo ocorrem na mesma barra, o padrão é o resultado adverso.
- Não há trailing stop ou break-even no EA fornecido; portanto não foram inventados.
- O EA pode decidir sobre valores H1/D1 em formação; a aproximação ponto-no-tempo é registrada nesta seção.

## 2. Ambiente

Kernel **Python (wdo-backtest)** (`conda env create -f environment.yml`; o pacote `wdo` é instalado em modo editável). Não execute com dados não auditados.


In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display
from wdo import (
    Config, load_mt5_export, load_csv_or_parquet, load_http_ohlcv, build_continuous_contract,
    run_backtest, print_backtest_report, plot_results, metrics,
)

# Caminhos relativos à raiz do repositório (o notebook vive em notebooks/).
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
assert (ROOT / 'src' / 'wdo').exists(), 'Execute a partir da raiz do repositório ou de notebooks/.'


## 3. Parâmetros

O cenário vem de `configs/wdo_default.toml`. Custos e valor do ponto são premissas configuráveis — não foram extraídos do EA.


In [ ]:
config = Config.from_toml(ROOT / 'configs' / 'wdo_default.toml')
START, END = '2026-01-01', '2026-09-01'
config


## 4. Data loader e contrato contínuo

A fonte deve conter `datetime, open, high, low, close, volume`; para rollover, também `contract`. A tabela `rollover` requer `contract, start, end`, sem sobreposição. APIs só são aceitas quando comprovam OHLCV M5 por vencimento.

In [ ]:
# Fonte ativa: exportação bruta do MT5 (wdo_data.csv), série já construída fora deste projeto.
# ATENÇÃO: origem/ajuste de rollover ainda não auditados (ver docs/agentic_documentation/08).
bars = load_mt5_export(ROOT / 'data' / 'raw' / 'wdo_data.csv', config.timezone)
print(f'{len(bars):,} barras M5 | {bars.datetime.min()} -> {bars.datetime.max()}')

# Opção A: série por contrato + calendário explícito (recomendado)
# raw = pd.read_parquet('wdo_m5_por_contrato.parquet')
# rollover = pd.read_csv('wdo_rollover.csv')
# bars = build_continuous_contract(raw, rollover, START, END, config.timezone)

# Opção B: arquivo já construído sob a mesma convenção documentada
# bars = load_csv_or_parquet('wdo_continuo_m5.parquet', config.timezone)

# Opção C: endpoint HTTP que devolva JSON de barras M5
# bars = load_http_ohlcv('https://SEU_ENDPOINT_M5_POR_CONTRATO', config.timezone)


## 5. Replay, trade log e métricas

O motor atualiza indicadores com dados anteriores, processa posição/pendências e avalia sinais em ordem cronológica. A chamada abaixo só deve ocorrer depois de definir `bars`.

In [ ]:
results = run_backtest(bars, start=START, end=END, initial_capital=10_000, config=config)
print_backtest_report(results)
trades = results.trades
events = results.events
display(trades.head(10))
display(events.head(20))


## 6. Visualizações

Equity, drawdown, PnL acumulado, distribuição, PnL mensal e trades mensais são produzidos por `plot_results`. A comparação LONG/SHORT é auditável no log e no resumo abaixo.

In [ ]:
plot_results(results)
long_short = trades.groupby('side').agg(trades=('trade_id','size'), net_pnl=('net_pnl','sum'), win_rate=('net_pnl', lambda x: x.gt(0).mean()*100))
display(long_short)


## 7. Validações e diferenças frente ao MT5

Antes de confiar nos resultados, valide: cobertura e timezone do rollover; barras ausentes; `trades` contra `equity`; política quando TP/SL coexistem na barra; e, se houver dados de tick, a divergência versus MT5. O resultado é um backtest OHLC conservador, não um relatório oficial de execução da corretora.

In [ ]:
# Reconciliação após a execução:
assert abs(trades.equity_after_trade.iloc[-1] - (config.initial_capital + trades.net_pnl.sum())) < 1e-6
assert trades.trade_id.is_unique
assert not bars.datetime.duplicated().any()
pd.Series(metrics(results)).to_frame('valor')
